# Session 4 — Statistical models & academic outputs
**90 minutes**

### Goals
1. Build a **small** Mean (SD) table suitable for a slide.
2. Run a transparent two-group comparison with effect size.
3. Fit one simple regression and read the summary without mystique.
4. Leave with a library roadmap (not a zoo of wrappers).

Calculations are inline. We use `scipy` / `pingouin` / `statsmodels` directly.


In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Readable plots for projection
plt.rcParams.update({
    "figure.figsize": (7, 4),
    "axes.titlesize": 13,
    "axes.labelsize": 11,
    "xtick.labelsize": 10,
    "ytick.labelsize": 10,
})

SESSION_DIR = Path.cwd()
WORKSHOP_DIR = SESSION_DIR.parent if SESSION_DIR.name == "sessions" else Path("workshop")
sys.path.insert(0, str(WORKSHOP_DIR))
from analysis.paths import data_path, stimuli_path
from scipy import stats
import pingouin as pg
import statsmodels.formula.api as smf


## 1. Analysis-ready table from GSR metrics

We use **Average_GSR** on the two search TOIs (`search_fearful` vs `search_nonfearful`).  
That contrast is interpretable; AOI-specific dwell columns are mostly empty outside name screens.


In [ ]:
gsr = pd.read_csv(data_path("tobii_gsr_demo", "Tobii_Pro_Lab_GSR_Demo_Project_Metrics.tsv"), sep="\t")
for c in ["Average_GSR", "Number_of_SCR", "Average_whole-fixation_pupil_diameter"]:
    gsr[c] = pd.to_numeric(gsr[c], errors="coerce")

df = gsr.loc[
    gsr["TOI"].isin(["search_fearful", "search_nonfearful"]),
    ["Participant", "TOI", "Media", "Average_GSR", "Number_of_SCR", "Average_whole-fixation_pupil_diameter",
     "Self-reported_animal_phobia"],
].copy()
df = df.rename(columns={"Average_whole-fixation_pupil_diameter": "pupil"})
df = df.dropna(subset=["Average_GSR"])
print("N rows:", len(df))
df.head(5)

## 2. Descriptive table — Mean (SD) by TOI

In [ ]:
desc = (
    df.groupby("TOI")["Average_GSR"]
    .agg(N="count", Mean="mean", SD="std")
    .reset_index()
)
desc["Mean (SD)"] = desc.apply(lambda r: f"{r['Mean']:.2f} ({r['SD']:.2f})", axis=1)
desc_slide = desc[["TOI", "N", "Mean (SD)"]]
desc_slide

In [ ]:
fig, ax = plt.subplots()
ax.bar(desc["TOI"].astype(str), desc["Mean"], yerr=desc["SD"], capsize=4, color="#345995")
ax.set_ylabel("Average_GSR")
ax.set_title("Mean GSR by search TOI (± SD)")
plt.xticks(rotation=15, ha="right")
plt.tight_layout()
plt.show()

## 3. Two-group comparison (Welch t-test)

Compare `search_fearful` vs `search_nonfearful` on Average_GSR.  
Report Welch’s df when scipy provides it — do not invent `n1+n2-2` for Welch.


In [ ]:
a = df.loc[df["TOI"] == "search_fearful", "Average_GSR"].dropna()
b = df.loc[df["TOI"] == "search_nonfearful", "Average_GSR"].dropna()

t_res = stats.ttest_ind(a, b, equal_var=False, alternative="two-sided")
nx, ny = len(a), len(b)
pooled = np.sqrt(((nx - 1) * a.std(ddof=1) ** 2 + (ny - 1) * b.std(ddof=1) ** 2) / (nx + ny - 2))
d = (a.mean() - b.mean()) / pooled if pooled > 0 else np.nan
welch_df = float(getattr(t_res, "df", np.nan))

p = float(t_res.pvalue)
p_txt = "< .001" if p < 0.001 else f"= {p:.3f}".replace("0.", ".")
apa = f"t({welch_df:.1f}) = {t_res.statistic:.2f}, p {p_txt}, d = {d:.2f}"

result = pd.DataFrame([
    {"group": "search_fearful", "N": nx, "Mean": round(a.mean(), 3), "SD": round(a.std(ddof=1), 3)},
    {"group": "search_nonfearful", "N": ny, "Mean": round(b.mean(), 3), "SD": round(b.std(ddof=1), 3)},
])
print(apa)
result

In [ ]:
# Cross-check with pingouin
pg.ttest(a, b, correction=True)

## 4. Simple regression (illustrative)

Model: `Number_of_SCR ~ Average_GSR + pupil` on the same search rows.  
Coefficients are for reading practice — not a causal claim.


In [ ]:
model_df = df.dropna(subset=["Average_GSR", "pupil", "Number_of_SCR"]).copy()
fit = smf.ols("Number_of_SCR ~ Average_GSR + pupil", data=model_df).fit()
coef_tbl = fit.summary2().tables[1][["Coef.", "Std.Err.", "t", "P>|t|"]].round(3)
coef_tbl

In [ ]:
fit_stats = pd.DataFrame([
    {"stat": "N", "value": int(fit.nobs)},
    {"stat": "R-squared", "value": round(fit.rsquared, 3)},
    {"stat": "Adj. R-squared", "value": round(fit.rsquared_adj, 3)},
])
fit_stats

### Optional second slide: phobia group on fearful search only


In [ ]:
fear = df.loc[df["TOI"] == "search_fearful"].dropna(subset=["Self-reported_animal_phobia", "Average_GSR"])
ph = (
    fear.groupby("Self-reported_animal_phobia")["Average_GSR"]
    .agg(N="count", Mean="mean", SD="std")
    .reset_index()
    .round(3)
)
ph

## 5. Library roadmap (discussion, 10 min)

**Core this week:** numpy, pandas, matplotlib, scipy, pingouin, statsmodels, Pillow, openpyxl  

**Recognize later:** neurokit2 (EDA/GSR pipelines), vendor SDKs, PyGaze, EyeLink toolkits  

**Reproducibility:** never overwrite raw exports; write analysis-ready CSVs; document I-VT thresholds and AOI definitions.


## Capstone (remaining time)
In pairs, produce **one** bar chart and **one** APA-like sentence using either:
- AOI dwell from Session 2, or
- Search TOI GSR from this session

### Exit ticket
Figure idea + statistical sentence + which library computed it.
